# مقایسه معماری‌ها — بنچمارک latent / reconstruction / prediction (Colab T4)

تولیدشده توسط `scripts/benchmarks/make_colab_notebook.py` — **دستی ویرایش نکنید**.
بعد از هر تغییر در `scripts/benchmarks/*`:
`python scripts/benchmarks/make_colab_notebook.py`

**پیش‌نیازها**
1. Runtime → Change runtime type → **T4 GPU**
2. ریپو `Mahdikalantari555/thesis_implementation` public و push شده باشد
3. در Drive: `MyDrive/thesis_colab/thesis_data_full_v2.zip` (~۲۹۴ MB) — همان زیپ، منبع
   رسمی v2؛ نسخهٔ قدیمی/جایگزین خوانده نمی‌شود.

**دو حالت، صریح جدا**
- `--fixture --smoke` (بخش ۹): فقط لوله‌بندی. هر manifest برچسب
  `SMOKE FIXTURE — plumbing evidence only, not a scientific result` می‌گیرد و هیچ ادعایی
  از آن استخراج نمی‌شود.
- حالت واقعی (بخش ۱۰): دیتای v2 روی T4. تخمین زمان از پروفایلِ خودِ اجرا می‌آید؛ سقف
  `budget.t4_hours` است و وقتی تمام شود سلول‌های مانده `Deferred` با ذکر دلیل می‌شوند.

هر منطقی که اینجا دیده می‌شود در `src/ssl_anomaly/benchmarking/*` یا
`scripts/benchmarks/*` زندگی می‌کند؛ نوتبوک فقط صدا می‌زند.

## ۱. محیط — وابستگی‌ها، clone، و مسیرهای کاری

In [ ]:
import os, sys, json, subprocess
import pathlib

REPO_URL = "https://github.com/Mahdikalantari555/thesis_implementation.git"
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "numpy", "scipy", "pandas", "scikit-learn", "pyyaml", "pyarrow"], check=True)
if not pathlib.Path("/content/thesis_implementation/.git").is_dir():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/content/thesis_implementation"], check=True)
os.chdir("/content/thesis_implementation")
sys.path.insert(0, "/content/thesis_implementation/src")
HEAD = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True,
                      text=True).stdout.strip()
BENCH, RUN = "/content/bench", "/content/drive/MyDrive/thesis_colab/bench_full"
pathlib.Path(BENCH).mkdir(parents=True, exist_ok=True)
pathlib.Path(RUN).mkdir(parents=True, exist_ok=True)
print("HEAD:", HEAD, "| cwd:", os.getcwd())
assert HEAD, "clone failed"

## ۲. سخت‌افزار — device و precision plan

In [ ]:
from ssl_anomaly.benchmarking import environment as env
from ssl_anomaly.benchmarking.config import load_benchmark_config

cfg = load_benchmark_config()
dev = env.detect_device(cfg, require=False).to_dict()
print(json.dumps(dev, indent=1, sort_keys=True, default=str))
GPU = bool(dev.get("cuda_available"))
if not GPU:
    print("GPU نیست — فقط بخش ۹ (fixture/smoke) مجاز است؛ بخش ۱۰ را اجرا نکنید.")

## ۳. دیتا — استخراج v2 و pin کردن هشِ همان فایل‌هایی که خوانده می‌شوند

In [ ]:
import pathlib, shutil, zipfile
from google.colab import drive

drive.mount("/content/drive")
ZIP = pathlib.Path("/content/drive/MyDrive/thesis_colab/thesis_data_full_v2.zip")
assert ZIP.is_file(), f"zip not found at {ZIP} (put it in MyDrive/thesis_colab/)"
shutil.copy(ZIP, "/content/data_v2.zip")
with zipfile.ZipFile("/content/data_v2.zip") as z:
    names = z.namelist()
    z.extractall(REPO_DIR + "/")
print("extracted entries:", len(names))

# هش‌ها همین‌جا محاسبه و pin می‌شوند — قرارداد مقدار از پیش‌نوشته‌شده را قبول نمی‌کند.
from ssl_anomaly.benchmarking import bindings as bind

rep = bind.verify_bindings(cfg, base=pathlib.Path(REPO_DIR), matrix_sha256=None,
                           applicability_sha256=None, compute_hashes=True,
                           strict=False).to_dict()
print(json.dumps({k: rep[k] for k in rep if k in ("ok", "blocking", "warnings", "artifacts")},
                 indent=1, sort_keys=True, default=str)[:2500])
assert not rep.get("blocking"), "artifact bindings are blocking — fix before any training"

## ۴. پیکربندی — قرارداد منجمد و ماتریس تعهدشده

In [ ]:
from ssl_anomaly.benchmarking import matrix as M
from ssl_anomaly.benchmarking.config import applicability_sha256, load_applicability, validate_config

app = load_applicability()
print("config sha:", cfg.document_sha256[:16], "| applicability sha:", applicability_sha256()[:16])
problems = validate_config(cfg.raw)
assert problems == [], problems
cells = M.build_matrix(cfg, app)
v = M.verify_committed_matrix(cfg, app)
print("committed matrix ok:", v["ok"], "| cells:", len(cells), "|", M.status_counts(cells))
assert v["ok"], v["problems"]

## ۵. split — فقط split‌های مجاز؛ test/monitoring هرگز

In [ ]:
from ssl_anomaly.benchmarking.splits import TestSplitAccessError, assert_admissible

ev = cfg.raw["evaluation"]
print("allowed:", ev["allowed_splits"], "| forbidden:", ev["forbidden_splits"])
assert_admissible(tuple(ev["allowed_splits"]))
try:
    assert_admissible(("train", "test"))
    raise SystemExit("LEAK GUARD BROKEN: a forbidden split was admitted")
except TestSplitAccessError as exc:
    print("guard holds:", str(exc)[:180])

## ۶. ناهنجاری مصنوعی — پنل ارزیابی و آزمون واقع‌نمایی

In [ ]:
import numpy as np
from ssl_anomaly.benchmarking import synthetic as syn
from ssl_anomaly.benchmarking.runner import load_clean_data

frame = load_clean_data(cfg, base=pathlib.Path(REPO_DIR), fixture=True)
panel = syn.apply_panel(frame, cfg, seed=int(cfg.raw["synthetic"]["eval_generator"]["seed"]),
                        pool=str(cfg.raw["synthetic"]["eval_generator"].get("pool",
                                                                            "dev_cycles_only")),
                        role="evaluation")
print(json.dumps(syn.audit_realism(panel, cfg), indent=1, sort_keys=True,
                 default=str)[:1800])

## ۷. ظرفیت و objective — باند ±۲۰٪ و ممیزی تفکیک هدف‌ها

In [ ]:
from ssl_anomaly.benchmarking.capacity import capacity_table, resolve_capacities
from ssl_anomaly.benchmarking.interface import assert_objectives_distinct

cap = resolve_capacities(cfg, d_channels=int(cfg.raw["training"]["channels"]), d_cov=10)
print("all in band:", cap["all_in_band"], "| spread:", cap["spread"]["max_over_min"])
print(capacity_table(cap))
assert cap["all_in_band"], {k: a["note"] for k, a in cap["arms"].items() if not a["in_band"]}

## ۸. جستجوی هایپرپارامتر — بودجهٔ برابر، انتخاب فقط روی validation

`d_model` در این جستجو **نیست**: باند ظرفیت را `capacity.tuning` انتخاب می‌کند و اگر HPO
هم همان را تکان می‌داد، برنده می‌توانست فقط «بزرگ‌تر بودن» باشد. برای دیتای واقعی
`--fixture` را بردارید.

In [ ]:
%%time
r = subprocess.run([sys.executable, "scripts/benchmarks/run_hpo.py", "--trials", "2",
                    "--epochs", "1", "--fixture", "--smoke", "--out", "/content/hpo"],
                   capture_output=True, text=True)
print((r.stdout or "")[-1500:] or (r.stderr or "")[-1500:])
print("rc:", r.returncode, "— selection در /content/hpo/hpo_selection.json؛ پیکربندی برنده با "
      "--selection-file به اجرا اضافه می‌شود و قرارداد را بازنویسی نمی‌کند")

## ۹. غربالگری — fixture/smoke: لوله‌بندی، نه نتیجه

In [ ]:
%%time
r = subprocess.run([sys.executable, "scripts/benchmarks/run_benchmark.py", "--execute", "--yes",
                    "--fixture", "--smoke", "--tier", "screening", "--block", "screening",
                    "--out", BENCH], capture_output=True, text=True)
print((r.stdout or "")[-2500:] or (r.stderr or "")[-2500:])
assert r.returncode == 0, f"smoke run failed rc={r.returncode}"

## ۱۰. بنچمارک کامل — واقعی، resumeدار، با سقف T4

رسمی‌ترین سلول نوتبوک. پیش‌شرط‌ها: سلول ۳ هش‌ها را pin کرده باشد و سلول ۴
`committed matrix ok: True` داده باشد. خروجی مستقیم در Drive نوشته می‌شود تا با قطع
session از دست نرود.

In [ ]:
%%time
cmd = [sys.executable, "scripts/benchmarks/run_benchmark.py", "--execute", "--yes",
       "--tier", "full", "--block", "full_benchmark", "--seeds", "13,42,2026",
       "--data-root", REPO_DIR, "--out", RUN, "--resume"]
if pathlib.Path("/content/hpo/hpo_selection.json").is_file():
    cmd += ["--selection-file", "/content/hpo/hpo_selection.json"]
print(" ", " ".join(cmd[1:]), flush=True)
with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True) as p:
    for line in p.stdout:
        print(line, end="", flush=True)
    rc = p.wait()
print("rc:", rc)

## ۱۱. checkpoint‌ها — هر run dir چه باید داشته باشد

In [ ]:
from ssl_anomaly.benchmarking import checkpoints as ck

def audit_run_dir_tree(root):
    rows = []
    for d in sorted(pathlib.Path(root).glob("runs/*")):
        if not d.is_dir():
            continue
        a = ck.audit_run_dir(d)
        rows.append({"dir": d.name, "ok": a["ok"], "missing": a["missing"],
                     "resume_point": ck.has_resume_point(d)})
    return rows

for label, root in (("smoke", BENCH), ("full", RUN)):
    rows = audit_run_dir_tree(root)
    bad = [r for r in rows if not r["ok"]]
    print(label, "dirs:", len(rows), "| incomplete:", len(bad))
    for r in bad[:5]:
        print("   ", r)
print("قاعده: bundle کامل است یا نقص — نقص را ترمیم کن یا Deferred کن؛ آپلود به‌عنوان کامل ممنوع")

## ۱۲. ادامه بعد از قطع — resume با تطبیق bindings

In [ ]:
r = subprocess.run([sys.executable, "scripts/benchmarks/run_benchmark.py", "--execute", "--yes",
                    "--fixture", "--smoke", "--tier", "screening", "--block", "screening",
                    "--out", BENCH, "--resume"], capture_output=True, text=True)
print((r.stdout or "")[-1500:] or (r.stderr or "")[-1500:])
print("resume فقط با hash یکسان مجاز است؛ اختلاف ⇒ CheckpointError که هر دو مقدار را چاپ می‌کند")

## ۱۳. ارزیابی روی پنل مصنوعی — family × severity، نه یک عدد تکی

In [ ]:
rows = []
for d in sorted(pathlib.Path(BENCH).glob("runs/*"))[:8]:
    man = json.loads((d / "run_manifest.json").read_text())
    macro = (man.get("metrics") or {}).get("macro") or {}
    rows.append({"exp": man["experiment_id"], "rung": man.get("ladder_rung"),
                 "stamp": (man.get("stamp") or "")[:22], "macro_auprc": macro.get("macro"),
                 "n_cells": macro.get("n_cells"), "cells_missing": macro.get("cells_missing")})
print(json.dumps(rows, indent=1)[:2500])

## ۱۴. شاخص‌های محیطی — مرجع، نه ground truth

In [ ]:
from ssl_anomaly.benchmarking import indices, runner as b_runner

# The frame the agreement is computed on is named beside every row: a fixture frame and
# the real v2 archive are different claims, and a table that hid which one it used would
# be a provenance defect.
real_ok = (pathlib.Path(REPO_DIR) / "data/processed/cycles_490_v2.parquet").is_file()
env_frame = b_runner.load_clean_data(cfg, base=pathlib.Path(REPO_DIR), fixture=not real_ok)
print("env frame data_source:", env_frame.data_source, "| fixture:", not real_ok)

n = int(env_frame.n)
is_dev = np.asarray(env_frame.split).astype(str) != "train"
fit_idx, scored = np.flatnonzero(~is_dev), np.flatnonzero(is_dev)
scores = indices.compute_all(env_frame.x, env_frame.channel_names, weather=None,
                             fit_idx=fit_idx, scored_idx=scored)
for name, idx in scores.items():
    print(json.dumps({"index": name, "available": idx.available, "missing": list(idx.missing),
                      "note": (idx.note or "")[:120]}, sort_keys=True))
print("CWSI_ET عمداً unavailable است: آرشیو جملهٔ مقاومت آیرودینامیکی را ندارد و جایگزینی‌اش "
      "با CWSI_LST یک شاخص دما را با نام ET معرفی می‌کرد")

# Task 9.18 in the contract's own form: arm-vs-index agreement with the
# block_bootstrap_by_field_year uncertainty. The arm score comes from the most recent
# run's bin_scores.npz (eval rows, self-describing); the year axis is the archive-year
# block derived from start_days, and the record states the block semantics — a calendar
# name would need the epoch sidecar, which is not part of the v2 bundle yet.
npz = None
for root in (pathlib.Path(RUN), pathlib.Path(BENCH)):
    if not root.is_dir():
        continue
    hits = sorted(root.glob("runs/*/bin_scores.npz"))
    if hits:
        npz = hits[-1]
        break
if npz is None:
    print("no bin_scores.npz yet — run section 9 or 10 first; agreement rows follow the run")
else:
    d = np.load(npz, allow_pickle=False)
    arm_bins, field_ids, split = d["bin_scores"], d["field_ids"], d["split"]
    assert (split.astype(str) == "dev").all(), "bin_scores.npz rows are the eval partition only"
    assert int(field_ids.size) == arm_bins.shape[0] == int(scored.size), (
        "run frame and env frame disagree; agreement rows would be misaligned")
    farms = np.repeat(np.asarray(field_ids), arm_bins.shape[1])
    start = np.asarray(env_frame.start_days)[scored]
    year_block = ((start - float(start.min())) // 365.25).astype(int)
    year_block = np.repeat(year_block, arm_bins.shape[1])
    cols = np.tile(np.arange(arm_bins.shape[1]), arm_bins.shape[0])
    lin = np.repeat(np.arange(arm_bins.shape[0]), arm_bins.shape[1]) * arm_bins.shape[1] + cols
    arm_flat = arm_bins.reshape(-1)[lin]
    caveat = str(cfg.raw["evaluation"]["environmental"]["caveat"])
    print("caveat:", caveat[:160], "...")
    for name, idx in scores.items():
        if not idx.available or idx.scores is None:
            print(json.dumps({"index": name, "agreement": "unavailable",
                              "reason": "index not computable on this archive"}, sort_keys=True))
            continue
        ag = indices.index_agreement(arm_flat, idx.scores[scored].reshape(-1)[lin],
                                     farm_ids=farms, years=year_block, n_boot=500)
        print(json.dumps({"index": name, **{k: ag.get(k) for k in
                            ("available", "spearman", "spearman_ci", "kendall",
                             "resampled_clustered_by", "n_blocks")}}, sort_keys=True))

## ۱۵. تعمیم — V1 زمانی، V2 مزرعه‌ای، V3 فضایی-زمانی

In [ ]:
from ssl_anomaly.benchmarking import protocols

print(json.dumps(protocols.audit_protocols(frame, cfg), indent=1, sort_keys=True,
                 default=str)[:2200])

## ۱۶. تحلیل آماری — bootstrap خوشه‌ای روی مزرعه، Wilcoxon، Friedman/Nemenyi، Holm

تعداد بوت‌استراپ از کف از پیش ثبت‌شده کمتر نمی‌شود؛ اگر کمتر باشد جدول `—` چاپ می‌کند،
نه برچسب «معنادار».

In [ ]:
r = subprocess.run([sys.executable, "scripts/benchmarks/report_benchmark.py", "--out", BENCH,
                    "--json"], capture_output=True, text=True)
print((r.stdout or "")[-1800:] or (r.stderr or "")[-1800:])

## ۱۷. پروفایل محاسباتی — ساعت T4، پارامتر، زمان inference

In [ ]:
for root in (BENCH, RUN):
    p = pathlib.Path(root) / "run_summary.json"
    if not p.is_file():
        print(root, "— no run_summary.json yet")
        continue
    s = json.loads(p.read_text())
    print(root, json.dumps({"wall": s["run"]["wall_seconds"], "by_status": s["by_status"],
                            "budget": s["run"]["budget"]}, indent=1, sort_keys=True,
                           default=str)[:900])

## ۱۸. سه گزارش برنده — detection / generalization / Pareto

In [ ]:
txt = pathlib.Path(BENCH) / "REPORT.md"
if txt.is_file():
    body = txt.read_text()
    i = body.find("## 3.")
    print(body[i:i + 3000])
else:
    print("no REPORT.md yet — run section 16 first")

## ۱۹. گزارش نهایی + evidence pack (CSV/JSON) در Drive

In [ ]:
r = subprocess.run([sys.executable, "scripts/benchmarks/report_benchmark.py", "--out", RUN,
                    "--copy-to", RUN + "/report"], capture_output=True, text=True)
print((r.stdout or "")[-1200:] or (r.stderr or "")[-1200:])

## ۲۰. خروجی — bundle کامل‌یا‌نقص‌دار با SHA256SUMS

`--collect` اول audit می‌کند و در صورت نقص **refuse** می‌کند (exit 2). بعد از آن تنها
راه استفاده، `sha256sum -c SHA256SUMS.txt` است.

In [ ]:
r = subprocess.run([sys.executable, "scripts/benchmarks/bundle_benchmark.py", RUN, "--collect",
                    "--dest", "/content/drive/MyDrive/thesis_colab/bundles"],
                   capture_output=True, text=True)
print((r.stdout or "")[-2200:] or (r.stderr or "")[-2200:])
assert r.returncode == 0, "bundle is a DEFECT — repair the cells or record them as Deferred"